# LYRA-Lite — 00 · Overview & Reproducibility

**A controlled study of order-sensitivity in the NLP architecture ladder.** We use a
non-linguistic token stream (single-cell gene-expression profiles) as an *instrument* to isolate
what sequence order contributes to a classifier — a question natural language cannot answer
cleanly, because it entangles order with meaning.

Front door: the research question, how to reproduce the pipeline, and the environment. No analysis
here — heavy logic lives in `src/lyra_lite/` and `scripts/`.

## 1. Research question

Sequence models (LSTM, Transformer) assume **order carries information**. Language can't test that
assumption cleanly — you can't remove word order without destroying meaning. A cell is an
**order-free set of gene tokens** with an explicit *(id, value)* channel and an
experimenter-imposed position, so it *can*. Two questions:

1. **Q1 — Does sequential inductive bias help when the signal is a set?**
   The architecture ladder FNN (bag-of-tokens) → LSTM (imposed order) → Transformer
   (permutation-invariant set), over rank-value gene tokens.
2. **Q2 — When we impose an arbitrary order, do order-sensitive models exploit it?**
   The ordering ablation — rank / random, **LSTM ascending vs. descending**
   (a recency probe), importance-ordering, and Transformer positional-encoding on/off.

The **balanced** blast-vs-normal task is *saturated* (a plain FNN reaches AUROC ≈ 0.99), so it
cannot rank architectures. We add a **rare-class stress test** — a controllable class-imbalance
regime (positive rate driven to 1 % / 0.1 %) — as the discriminative regime where inductive biases
separate. This is the NLP analogue of rare-class / rare-intent detection: a methods knob, not a
clinical claim.

## 2. Reproduce the pipeline

```bash
# 1. one-time: build the cached dataset artifact (~15 min, then instant)
python scripts/train.py data=scpca

# 2. train an architecture (writes best_model.pt + metrics.json to outputs/<date>/<time>/)
python scripts/train.py data=scpca model=fnn                     # or model=lstm / model=transformer
python scripts/train.py -m data=scpca model=fnn seed=1,2,3,4,5   # multi-seed for CIs

# 3. rare-class stress-test eval on a finished run (script/artifacts keep the mrd_* name in code)
python scripts/evaluate_mrd.py --run_dir outputs/<date>/<time>

# 4. figures: notebooks 01-05 read the artifacts above and plot
```

In [1]:
# Environment & versions (reproducibility appendix)
import platform
from importlib.metadata import version, PackageNotFoundError

print("python  :", platform.python_version())
for pkg in ["numpy", "pandas", "scikit-learn", "torch", "scanpy", "anndata", "hydra-core", "matplotlib"]:
    try:
        print(f"{pkg:13s}:", version(pkg))
    except PackageNotFoundError:
        print(f"{pkg:13s}: (not installed)")

try:
    import torch
    dev = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
    print("device  :", dev)
except Exception:
    pass

python  : 3.12.0
numpy        : 2.4.6
pandas       : 3.0.3
scikit-learn : 1.9.0
torch        : 2.12.0
scanpy       : 1.11.5
anndata      : 0.11.4
hydra-core   : 1.3.2
matplotlib   : 3.11.0
device  : mps


## 3. Repository map

```
src/lyra_lite/   data (scpca loader + cache, representation), models, analysis (ece, mrd_eval), training
scripts/         train.py, evaluate_mrd.py, aggregate.py
configs/         Hydra: data/ model/ training/ eval/
data/cache/      cached dataset artifacts (.npz) + provenance manifests (.json)   [gitignored]
outputs/         per-run checkpoints, metrics.json, mrd/ results                   [gitignored]
notebooks/       this analysis layer -> figures/ + tables/ for the paper
```

## 4. Notebook index

| # | Notebook | Main Output |
|---|----------|--------|
| 01 | dataset & cohort (the corpus) | Table 1 (cohort) |
| 02 | exploratory data analysis (corpus statistics) | Fig. 1 (group shift) |
| 03 | representation & tokenization | Methods · representation |
| 04 | architecture benchmark (Q1: ladder, balanced + rare-class) | inductive-bias baseline |
| 05 | ordering ablation (Q2: order & positional bias) | Fig. 2 (ordering / asc-desc) |

## 5. Reproducibility commitments

- All randomness seeded (`cfg.seed`); the dataset build is seed-independent, so one cache serves every run.
- **Patient-level** train/val/test split (`GroupShuffleSplit` on `participant_id`) — no patient in two splits.
- Model selection on **validation loss**; the test set is touched once.
- Every run snapshots its config to `outputs/<...>/.hydra/`; `evaluate_mrd.py` reads that snapshot, so eval always matches the checkpoint.